<a href="https://colab.research.google.com/github/AIN-ELCTRA/skills-introduction-to-github/blob/main/hotel%20chat%20bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import re
from datetime import datetime
from dateutil import parser as date_parser

# ----- spaCy import & compatibility handling -----
try:
    import spacy
    from spacy.matcher import Matcher
except TypeError as e:
    # Typical issue: incompatible pydantic / spaCy versions
    print("❌ spaCy / pydantic compatibility error:")
    print(e)
    print(
        "Please reinstall spaCy with a compatible pydantic version.\n"
        "Example:\n"
        "  pip install 'pydantic<2.0.0' 'spacy<3.7'\n"
        "  python -m spacy download en_core_web_sm"
    )
    sys.exit(1)
except ImportError:
    print("❌ spaCy is not installed. Install it with:")
    print("  pip install spacy\n  python -m spacy download en_core_web_sm")
    sys.exit(1)


def load_nlp():
    """Load spaCy model, download if needed."""
    try:
        nlp = spacy.load("en_core_web_sm")
    except OSError:
        from spacy.cli import download

        print("Downloading spaCy model 'en_core_web_sm'...")
        download("en_core_web_sm")
        nlp = spacy.load("en_core_web_sm")
    return nlp


nlp = load_nlp()
matcher = Matcher(nlp.vocab)


# ----- PATTERN SETUP -----
def setup_patterns(matcher: Matcher):
    """Define patterns for greetings, names, guests, breakfast yes/no."""
    # GREET: hello, hi, hey, good morning, good evening...
    greet_patterns = [
        [{"LOWER": {"IN": ["hi", "hello", "hey"]}}],
        [{"LOWER": "good"}, {"LOWER": {"IN": ["morning", "afternoon", "evening"]}}],
    ]
    matcher.add("GREET", greet_patterns)

    # NAME: crude pattern for a proper-noun name (e.g., "John", "John Smith")
    name_patterns = [
        [{"POS": "PROPN"}],
        [{"POS": "PROPN"}, {"POS": "PROPN"}],
    ]
    matcher.add("NAME", name_patterns)

    # GUESTS: number of guests (e.g., "2 guests", "3 people")
    guests_patterns = [
        [
            {"LIKE_NUM": True},
            {"LOWER": {"IN": ["guest", "guests", "people", "persons"]}},
        ]
    ]
    matcher.add("GUESTS", guests_patterns)

    # BREAKFAST_YES: with breakfast / include breakfast / yes breakfast
    breakfast_yes_patterns = [
        [
            {"LOWER": {"IN": ["with", "including", "include"]}},
            {"LOWER": "breakfast"},
        ],
        [{"LOWER": "yes"}, {"LOWER": "breakfast"}],
        [{"LOWER": "breakfast"}],  # if they just say "breakfast" when asked
    ]
    matcher.add("BREAKFAST_YES", breakfast_yes_patterns)

    # BREAKFAST_NO: without breakfast / no breakfast
    breakfast_no_patterns = [
        [
            {"LOWER": {"IN": ["without", "no"]}},
            {"LOWER": "breakfast"},
        ],
        [{"LOWER": "no"}, {"LOWER": "thanks"}],
        [{"LOWER": "no"}],  # when asked directly
    ]
    matcher.add("BREAKFAST_NO", breakfast_no_patterns)


setup_patterns(matcher)


# ----- GENERIC EXTRACTION HELPERS -----
def extract_entity(doc, entity_type: str):
    """Extract first entity of given spaCy label (e.g. PERSON, DATE)."""
    for ent in doc.ents:
        if ent.label_ == entity_type:
            return ent.text.strip()
    return None


def extract_match(doc, label: str):
    """Return text for the first match with the given Matcher label."""
    matches = matcher(doc)
    for match_id, start, end in matches:
        if nlp.vocab.strings[match_id] == label:
            span = doc[start:end]
            return span.text
    return None


def extract_name(text: str):
    """Try to extract the user's name via NER, fallback to NAME pattern."""
    doc = nlp(text)

    # 1) Try spaCy PERSON entity
    person = extract_entity(doc, "PERSON")
    if person:
        return person

    # 2) Fallback to NAME pattern
    name_match = extract_match(doc, "NAME")
    if name_match:
        return name_match.strip()

    return None


# ----- BASIC IO HELPERS -----
def ask(prompt: str) -> str:
    """Ask the user a question and return their answer."""
    return input(prompt).strip()


def exit_if_requested(user_input: str):
    """Exit gracefully if the user wants to quit."""
    if user_input.lower() in {"exit", "quit"}:
        print("Exiting booking process. Goodbye!")
        sys.exit(0)


def get_validated_input(prompt: str, validate_fn):
    """
    Ask until validate_fn(text) returns a non-None value.
    validate_fn should return either a parsed value or None.
    Typing 'exit' or 'quit' terminates the program.
    """
    while True:
        user_input = ask(prompt)
        exit_if_requested(user_input)

        value = validate_fn(user_input)
        if value is not None:
            return value
        print("I couldn't understand that. Please try again, or type 'exit' to quit.\n")


# ----- SPECIFIC VALIDATION FUNCTIONS -----
def validate_name(text: str):
    """Validate that the input looks like a name (and not just a greeting)."""
    doc = nlp(text)

    # Reject if it's just a greeting
    if extract_match(doc, "GREET"):
        print("That sounds like a greeting. Please enter your name instead.")
        return None

    name = extract_name(text)
    if not name:
        print("I couldn't detect a name in that input.")
        return None

    return name


def parse_date(text: str):
    """Parse a date string into a datetime.date, or return None on failure."""
    try:
        dt = date_parser.parse(text, fuzzy=True)
        return dt.date()
    except (ValueError, TypeError):
        return None


def validate_checkin_date(text: str):
    date_obj = parse_date(text)
    if not date_obj:
        print("I couldn't understand that date.")
        return None
    return date_obj


def validate_checkout_date_factory(checkin_date):
    """Return a validator that ensures checkout is after checkin."""

    def validate_checkout_date(text: str):
        date_obj = parse_date(text)
        if not date_obj:
            print("I couldn't understand that date.")
            return None
        if date_obj <= checkin_date:
            print("Check-out date must be after check-in date.")
            return None
        return date_obj

    return validate_checkout_date


WORD_NUMBERS = {
    "one": 1,
    "two": 2,
    "three": 3,
    "four": 4,
    "five": 5,
    "six": 6,
    "seven": 7,
}


def extract_guests(text: str):
    """Extract a guest count (1–7) from free text."""
    doc = nlp(text)

    # Try the GUESTS pattern
    guests_span = extract_match(doc, "GUESTS")
    if guests_span:
        # Expect something like "2 guests"
        m = re.search(r"\d+", guests_span)
        if m:
            count = int(m.group())
            if 1 <= count <= 7:
                return count

    # Try plain number in text
    m = re.search(r"\d+", text)
    if m:
        count = int(m.group())
        if 1 <= count <= 7:
            return count

    # Try word numbers: "two guests"
    lower_text = text.lower()
    for word, num in WORD_NUMBERS.items():
        if word in lower_text:
            if 1 <= num <= 7:
                return num

    return None


def validate_guests(text: str):
    """Validate that guest count is between 1 and 7."""
    guests = extract_guests(text)
    if guests is None:
        print("Please specify a number of guests between 1 and 7.")
        return None
    return guests


def extract_breakfast_choice(text: str):
    """
    Extract breakfast preference:
    - returns True  -> with breakfast
    - returns False -> without breakfast
    - None if unsure
    """
    doc = nlp(text)

    if extract_match(doc, "BREAKFAST_YES"):
        return True
    if extract_match(doc, "BREAKFAST_NO"):
        return False

    # Also check simple yes/no
    lower = text.lower()
    if any(word in lower for word in ["yes", "y"]):
        return True
    if any(word in lower for word in ["no", "n"]):
        return False

    return None


def validate_breakfast(text: str):
    choice = extract_breakfast_choice(text)
    if choice is None:
        print("Please answer yes or no regarding breakfast.")
        return None
    return choice


# ----- BOOKING LOGIC -----
ROOM_RATE_NO_BF = 80.0   # base example rate per night
ROOM_RATE_WITH_BF = 95.0 # example rate per night with breakfast


def collect_booking_info():
    """
    Walk through the whole dialogue:
    - name
    - check-in / check-out
    - number of guests
    - breakfast
    """
    print("\nWelcome to the Hotel Booking Chatbot!")
    print("I'll help you reserve a room step by step.")
    print("You can type 'exit' at any time to quit.\n")

    name = get_validated_input("First, what is your name?\n> ", validate_name)

    check_in = get_validated_input(
        f"Nice to meet you, {name}. When would you like to check in?\n> ",
        validate_checkin_date,
    )

    check_out = get_validated_input(
        "And when would you like to check out?\n> ",
        validate_checkout_date_factory(check_in),
    )

    guests = get_validated_input(
        "How many guests will be staying? (1–7)\n> ",
        validate_guests,
    )

    breakfast = get_validated_input(
        "Would you like to include breakfast? (yes/no)\n> ",
        validate_breakfast,
    )

    nights = (check_out - check_in).days
    rate = ROOM_RATE_WITH_BF if breakfast else ROOM_RATE_NO_BF
    total_cost = nights * rate

    booking = {
        "name": name,
        "check_in": check_in,
        "check_out": check_out,
        "nights": nights,
        "guests": guests,
        "breakfast": breakfast,
        "rate_per_night": rate,
        "total_cost": total_cost,
    }

    return booking


def confirm_booking(booking: dict):
    """Show booking summary, ask for confirmation, return True/False."""
    print("\n--- Booking Summary ---")
    print(f"Name:        {booking['name']}")
    print(f"Check-in:    {booking['check_in'].strftime('%Y-%m-%d')}")
    print(f"Check-out:   {booking['check_out'].strftime('%Y-%m-%d')}")
    print(f"Nights:      {booking['nights']}")
    print(f"Guests:      {booking['guests']}")
    print(
        "Breakfast:   "
        + ("Included" if booking["breakfast"] else "Not included")
    )
    print(f"Rate/night:  {booking['rate_per_night']:.2f} EUR")
    print(f"Total cost:  {booking['total_cost']:.2f} EUR")
    print("------------------------")

    while True:
        answer = ask("Do you want to confirm this booking? (yes/no)\n> ").lower()
        exit_if_requested(answer)

        if answer.startswith("y"):
            print("\n✅ Your booking has been confirmed. Thank you!")
            return True
        elif answer.startswith("n"):
            print("\n❌ Booking was cancelled.")
            return False
        else:
            print("Please answer 'yes' or 'no', or type 'exit' to quit.\n")


# ----- MAIN LOOP -----
def main():
    print("Hotel Booking Chatbot (offline console version)")
    print("Type 'hello' or 'hi' to start a booking, or 'exit' to quit.\n")

    while True:
        user_input = ask("> ")
        if not user_input:
            continue

        if user_input.lower() in {"exit", "quit"}:
            print("Goodbye!")
            break

        doc = nlp(user_input)
        if extract_match(doc, "GREET"):
            # Start full booking process
            booking = collect_booking_info()
            confirm_booking(booking)

            # Ask if they want another booking
            again = ask("\nWould you like to make another booking? (yes/no)\n> ").lower()
            if again.startswith("y"):
                print("\nOkay, let's start a new booking.\n")
                continue
            else:
                print("Thank you for using the Hotel Booking Chatbot. Goodbye!")
                break
        else:
            print("Please greet me with 'hello' or type 'exit' to quit.\n")


if __name__ == "__main__":
    main()


Hotel Booking Chatbot (offline console version)
Type 'hello' or 'hi' to start a booking, or 'exit' to quit.

> hi

Welcome to the Hotel Booking Chatbot!
I'll help you reserve a room step by step.
You can type 'exit' at any time to quit.

First, what is your name?
> nmom
I couldn't detect a name in that input.
I couldn't understand that. Please try again, or type 'exit' to quit.

First, what is your name?
> momo
Nice to meet you, momo. When would you like to check in?
> 21
And when would you like to check out?
> 22
How many guests will be staying? (1–7)
> 3
Would you like to include breakfast? (yes/no)
> yes

--- Booking Summary ---
Name:        momo
Check-in:    2025-12-21
Check-out:   2025-12-22
Nights:      1
Guests:      3
Breakfast:   Included
Rate/night:  95.00 EUR
Total cost:  95.00 EUR
------------------------
Do you want to confirm this booking? (yes/no)
> yes

✅ Your booking has been confirmed. Thank you!

Would you like to make another booking? (yes/no)
> no
Thank you for usi